# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Colab execution reached the analysis but failed: the PURR /serve/1 URL returned a 27 KB HTML response instead of the expected ZIP (cell 3 assertion: not a ZIP archive); downstream R analysis and metrics then failed. The notebook is preserved as an unverified draft. A later attempt has not yet produced a successful run result.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Quantifying physiological trait variation with automated hyperspectral imaging in rice.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-28f074ee4e709feeb203a3120b869fa0)


# Rice leaf nitrogen from hyperspectral imaging — RReliefF-PLSR (author R pipeline)

Reproduction of the strongest single result of:

> Ting T, Souza A, Imel RK, Guadagno CR, Hoagland C, Yang Y, Wang DR (2023).
> **Quantifying physiological trait variation with automated hyperspectral imaging in rice.**
> *Frontiers in Plant Science* 14:1229161. [doi:10.3389/fpls.2023.1229161](https://doi.org/10.3389/fpls.2023.1229161)

**Executed scope:** the author's `N_400wv_sideviewA.Rmd` pipeline — RReliefF selection of 400 wavelengths
from Week-9 side-view hyperspectral reflectance, treatment-balanced 80/20 calibration/validation split
(seed 1), orthogonal-scores PLSR with leave-one-out component selection, jackknife coefficients, and
calibration/validation metrics for **leaf nitrogen content N (%)**.
Paper reference values (abstract/Fig. 4): validation R² = 0.797, RMSEP = 0.264, n = 70 cal / 18 val.
A single-example/partial run like this does **not** verify the paper's dataset-level claims (SVM
classification, cross-group transfer, hyperparameter sweeps).

**Provenance / adaptation statement:** the author's R code is **executed directly** (not re-implemented);
Python cells only download inputs, install R packages, and launch `Rscript`. The pipeline is chunk-for-chunk
from `To-Chia/rice_imaging_ms` commit `b6b6220`; `f.coef.valid` is embedded verbatim from
`plantphys/spectratrait` @ `d159c5d` (GPL-3.0, © Julien Lamour). Material adaptations: hard-coded
`/content` paths, plots saved as PNG, and the figure annotations for %RMSEP are **computed from the model**
instead of the author's hard-coded values. Untested behavior is not guaranteed.
（作者のRコードをそのまま実行します。Pythonは入力ダウンロード・Rパッケージ導入・`Rscript`起動のみ。）

## Runtime selection — CPU
**Select Runtime → Change runtime type → CPU.** The method is classical multivariate statistics
(ReliefF + PLSR); the author code has no GPU path, so no GPU is used or needed.
Expected total ≈ 15–20 min (R package install ≈ 5–15 min, analysis ≈ 2–5 min), downloads ≈ 1 MB.
**Run all** to execute. 日本語: CPU ランタイムを選択し、「すべてのセルを実行」してください。
合計 15〜20 分程度、ダウンロードは約 1 MB です。

## 1. Check R availability
The Colab Python runtime ships with `Rscript`. This cell only verifies it (no science here).
日本語: Colab ランタイムに `Rscript` が用意されているかを確認します。

In [ ]:
import subprocess, sys
r = subprocess.run(["Rscript", "--version"], capture_output=True, text=True, check=True)
print(r.stderr.strip())

## 2. Install required R packages
Installs the author pipeline's dependencies from CRAN: `pls` (PLSR), `CORElearn` (RReliefF — compiled
C++, the slowest), `reshape2`, `dplyr`, `ggplot2`, `ggpubr`, `gridExtra`, `scales`. This streams the
install log; it takes roughly 5–15 minutes on a fresh CPU runtime. The author script also lists
`tidyverse`/`kableExtra` but does not use them in the executed path; `spectratrait` is embedded as a
verbatim function instead of a package install. 日本語: 著者パイプラインに必要な R
パッケージを CRAN から導入します（CORElearn は C++ コンパイルで数分）。

In [ ]:
import time
pkgs = "reshape2 dplyr ggplot2 pls ggpubr gridExtra scales CORElearn"
t0 = time.time()
subprocess.run(["Rscript", "-e",
                f'options(repos=c(CRAN="https://cloud.r-project.org")); '
                f'install.packages(strsplit("{pkgs}", " ")[[1]])'],
               check=True)
print(f"R package install finished in {time.time()-t0:.0f}s")

## 3. Download the author dataset bundle (Purdue PURR, CC BY 4.0)
The ground-truth and hyperspectral CSVs live in the paper's data bundle
**PURR publication 4079 v1, DOI [10.4231/SY0A-W347](https://doi.org/10.4231/SY0A-W347)**
(6 files, ~1 MB, CC BY 4.0). We download the archive once, verify it is a real ZIP, list members,
extract the two files the script reads, and record sizes + SHA-256.
日本語: 論文のデータバンドル（PURR 4079 v1, CC BY 4.0）を取得し、ZIP 検証・展開・
ハッシュ確認を行います。バンドル全体（約1MB）を取得します。

In [ ]:
import io, zipfile, hashlib, requests

BUNDLE_URL = "https://purr.purdue.edu/publications/4079/serve/1?render=archive"  # PURR 4079 v1 archive
NEEDED = {"phyraw_forHSI_2022.csv", "HSI_side_for_biotraits_mod_W9.csv"}

r = requests.get(BUNDLE_URL, timeout=120, allow_redirects=True)
r.raise_for_status()
blob = r.content
print(f"HTTP {r.status_code}, final URL {r.url}, {len(blob)} bytes")
assert blob[:2] == b"PK", "not a ZIP archive"

outer = zipfile.ZipFile(io.BytesIO(blob))
print(f"{len(outer.namelist())} outer members: {outer.namelist()}")
inner_name = [n for n in outer.namelist() if n.endswith("bundle.zip")]
assert len(inner_name) == 1, "expected one nested bundle.zip"
blob = outer.read(inner_name[0])
print(f"inner bundle: {len(blob)} bytes, sha256 = {hashlib.sha256(blob).hexdigest()}")
assert blob[:2] == b"PK", "inner bundle not a ZIP"

zf = zipfile.ZipFile(io.BytesIO(blob))
names = zf.namelist()
print(f"{len(names)} members: {names}")
assert all(any(n.endswith(f) for n in names) for f in NEEDED), "required CSVs missing"

import os
for member in names:
    base = os.path.basename(member)
    if base in NEEDED:
        dest = ("/content/data/ground_truth/" if base == "phyraw_forHSI_2022.csv"
                else "/content/data/hyperspectral_imaging/")
        os.makedirs(dest, exist_ok=True)
        data = zf.read(member)
        with open(dest + base, "wb") as fh:
            fh.write(data)
        print(f"extracted {base}: {len(data)} bytes, sha256={hashlib.sha256(data).hexdigest()[:16]}...")


## 4. The author R pipeline (written verbatim to `/content`)
Writes `n400_plsr.R`: a chunk-for-chunk copy of `N_400wv_sideviewA.Rmd`
(To-Chia/rice_imaging_ms @ `b6b6220`). The cell below shows the full source; the next cell runs it.
Mapping: data import & `wv>2500` trim → Rmd chunk `import data`; seed-1 treatment-balanced split →
chunk `data`; CORElearn `RReliefFexpRank` (400 wv) → chunk `feature selection`; OSC-PLSR + LOO +
`min_selectNcomp` → chunks `number of components CV`/`plsr output`; jackknife CI/PI via `f.coef.valid`
→ chunk `jackknife coef`. 日本語: 著者の Rmd をチャンク単位でそのまま `/content` に書き出します。

In [ ]:
%%writefile /content/n400_plsr.R
## Rice hyperspectral leaf-N pipeline (RReliefF-PLSR), Week 9 side-view
## Verbatim chunk-for-chunk adaptation of:
##   To-Chia/rice_imaging_ms @ b6b6220bc34f58d56ca5a9d6417393907a441da2
##   N_400wv_sideviewA.Rmd (Ting et al. 2023, Front. Plant Sci. 14:1229161)
## f.coef.valid embedded verbatim from:
##   plantphys/spectratrait @ d159c5dda9d6739105d95b357e48f7754daf4978
##   R/f.coef.valid.R (GPL-3.0, (c) Julien Lamour)
## Only changes: hard-coded /content paths, plots saved as PNG, results as CSV.

suppressMessages({
  library(dplyr)
  library(reshape2)
  library(ggplot2)
  library(pls)
  library(ggpubr)
  library(gridExtra)
  library(scales)
  library(CORElearn)
})

data_root <- "/content/data"
out_dir   <- "/content/outputs"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

## ---- import data (Rmd chunk `import data`) ----
phyraw <- read.csv(file.path(data_root, "ground_truth/phyraw_forHSI_2022.csv"), header = TRUE, sep = ",", dec = ".")
ref_meta <- read.csv(file.path(data_root, "hyperspectral_imaging/HSI_side_for_biotraits_mod_W9.csv"), header = TRUE, sep = ",", dec = ".")
ref_meta = ref_meta[ ,-c(746:762)] # remove wv > 2500

## ---- functions and settings (Rmd chunk, verbatim) ----
input_for_RReliefF = function(X_IDdata, y_metadata, cal.ID, y_name, num_X_names){
  y_cal = y_metadata[y_metadata$ID %in% cal.ID, ]
  X_cal = X_IDdata[X_IDdata$ID %in% cal.ID, ]

  data_ID <- merge(X_cal, subset(y_cal, select = -c(Treatment, Subpop)),
                     by = "ID")
  data <- data_ID[ ,-1] # drop ID
  names(data) = c(num_X_names, y_name)

  return(data)
}

RReliefF_algorithm = function(pre_cal, # calibration data with full spectrum
                             y_name, num_X_names,
                             estimator = "RReliefFexpRank", iterationtimes = 100){

    attr <- attrEval(y_name, pre_cal, estimator = estimator,
                   ReliefIterations = nrow(pre_cal)*iterationtimes)

    attr_long <- data.frame(value = unname(attr),
                            wv = num_X_names)

    return(attr_long)
}

cal_val_data_sets = function(cal.ID, val.ID,
                             X_IDdata, y_metadata, X_attr,
                             y_name, num_X_names,
                             plot=FALSE, y_unit=NULL){

  X_select <- as.matrix(X_IDdata[ ,as.character(X_attr$wv)])

  cal.plsr.data = data.frame(y_metadata,
                           Spectra=I(X_select))[y_metadata$ID %in% cal.ID, ]

  val.plsr.data = data.frame(y_metadata,
                           Spectra=I(X_select))[y_metadata$ID %in% val.ID, ]

  if (isTRUE(plot)) {
        par(mfrow=c(1,2))
    hist(cal.plsr.data[ ,y_name], main="cal", xlab=paste0(y_name, y_unit))
    hist(val.plsr.data[ ,y_name], main="val", xlab=paste0(y_name, y_unit),
         breaks = 3)}

 return(list(cal.plsr.data, val.plsr.data))
}

sel_wv_plotdata = function(data){
  data.df = as.data.frame(cbind(data$Spectra))
  names(data.df) = as.numeric(attr(data$Spectra, "dimnames")[[2]])
  data.df$ID = data$ID
  data.long = melt(data.df, id = "ID") %>%
  mutate(variable = as.numeric(as.character(variable)), .keep="unused")
  return(data.long)
}

quantile_sel_ref = function(data, lo_q, hi_q){
  summary = data %>%
  group_by(variable) %>%
  summarize(mean_value=mean(value),
            lo=quantile(value, lo_q),
            hi=quantile(value, hi_q))
  names(summary)=c("wv", "mean_ref", "lo_ref", "hi_ref")
  summary_long = melt(summary, id="wv")
  return(summary_long)
}

min_selectNcomp <- function(object, ncomp = maxComps, plot = FALSE, ...) {
    if (!isTRUE(object$validation$method == "CV"))
        stop("No cross-validation data available in model")
    ## check that Y is univariate
    if (dim(residuals(object))[2] > 1)
        stop("Only univariate response supported")
    ## includes zero component, the first element is the zero component
    rmseps <- c(RMSEP(object, "CV")$val)
    maxIdx <- ncomp + 1
    absBest <- which.min(rmseps[seq_len(maxIdx)])
    selection <- absBest -1
    if (is.null(origResponse <- object$y))

        origResponse <- c(predict(object, ncomp = 1) + residuals(object)[,1,1])

    allresids <- cbind(origResponse-
                         (sum(origResponse)-origResponse)/(length(origResponse) - 1),
                       object$validation$pred[,1,] - origResponse)
        ## include LOO prediction with zero components (i.e., the
        ## mean). For the mean we should also use the LOO estimate...
    residsds <- apply(allresids, 2, sd) / sqrt(nrow(allresids))
    uls <- rmseps - residsds
    lls <- rmseps + residsds
    if (absBest > 0) {
      if (isTRUE(plot)) {
        par(mfrow=c(1,1), mar=c(6, 7, 1, 0.4), oma=c(0, 0.1, 0, 0.2), mgp = c(5, 2, 0))
# the first index should be zero component, but the rmseps considered it as the first element
            xvals <- seq_along(rmseps) - 1
            plot(xvals, rmseps, ylab = "RMSEP",
                 xlab = "Number of components", type = "b", ...)
            arrows(xvals, uls, xvals, lls,
                   code = 3, col = "gray", angle = 90, length = .1)
            abline(h = rmseps[absBest], col = "gray", lty = 3)
            abline(v = absBest - 1, col = "blue", lty = 3)
        }
      selection
    } else {
        warning("Lowest CV error found at 0 components")
    }
}

plsr_results = function(model, nComps, cal.plsr.data, val.plsr.data, y_name){
  y_cal = cal.plsr.data[ ,y_name]
  y_cal_fit = model$fitted.values[,,nComps]
  PLSR_CV_Predicted = as.vector(model$validation$pred[,,nComps])
  PLSR_CV_Residuals = PLSR_CV_Predicted-y_cal
  #intercept = F, estimates for a model with zero component does not need to be returned
  cal.R2 <- round(pls::R2(model,intercept=FALSE)[[1]][nComps],3)
  cal.RMSEP <- round(sqrt(mean(PLSR_CV_Residuals^2)),3)
  cal_y_range <- range(y_cal)[2]-range(y_cal)[1]
  cal.NRMSEP <- round((sqrt(mean(PLSR_CV_Residuals^2))/cal_y_range)*100, 3)

  y_val = val.plsr.data[ ,y_name]
  y_val_fit = as.vector(
    predict(model,newdata = val.plsr.data,ncomp=nComps,type="response")[,,])
  y_val_resids =  y_val_fit-y_val
  val.R2 <- round(pls::R2(model,newdata=val.plsr.data,intercept=FALSE)[[1]][nComps],3)
  val.RMSEP <- round(sqrt(mean(y_val_resids^2)),3)
  val_y_range <- range(y_val)[2]-range(y_val)[1]
  val.NRMSEP <- round((sqrt(mean(y_val_resids^2))/val_y_range)*100,3)

  mm=data.frame(
    data_set=c("cal", "val"),
    R2 = c(cal.R2, val.R2),
    RMSEP = c(cal.RMSEP, val.RMSEP),
    NRMSEP = c(cal.NRMSEP, val.NRMSEP)
  )
  cal.plsr.output = data.frame(Observed = y_cal,
                               Predicted = PLSR_CV_Predicted,
                               Residuals = PLSR_CV_Residuals,
                               Treatment = cal.plsr.data$Treatment,
                               Subpop = cal.plsr.data$Subpop)
  val.plsr.output = data.frame(Observed = y_val,
                               Predicted = y_val_fit,
                               Residuals = y_val_resids,
                               Treatment = val.plsr.data$Treatment,
                               Subpop = val.plsr.data$Subpop)
  list = list(cal.plsr.output = cal.plsr.output,
              val.plsr.output = val.plsr.output,
              mm = mm)
  return(list)

}
`%notin%` <- Negate(`%in%`)

# plsr setting
pls::pls.options(plsralg = "oscorespls")
pls::pls.options("plsralg")

# plot spectra color settings
summary_ref_col = c("mean_ref" = "forestgreen", "lo_ref"="green", "hi_ref"="green")

# plot prediction color and shape settings
subpop_col = c("TRJ" = 'blue', 'IND' = 'red')
treatment_shape = c('N1' = 16, 'N2' = 1)

## ---- data (Rmd chunk `data`, verbatim incl. seed) ----
y_metadata <- phyraw %>%
  select("ID", "N", "Treatment", "Subpop") %>%
  mutate(ID = as.factor(ID)) %>%
  na.omit()

X_IDdata <- ref_meta %>%
  select(!c(Accession, Subpop, Country, Treatment)) %>%
  mutate(ID = as.factor(ID)) %>%
  filter(ID %in% y_metadata$ID)
num_X_names = as.numeric(gsub("_", ".", sub("...", "",  names(X_IDdata[,-1]))))
names(X_IDdata) = c("ID", as.character(num_X_names))
set.seed(1)
cal.ID <- y_metadata %>%
  select(ID, Treatment) %>%
  group_by_at(vars(all_of("Treatment"))) %>%
  slice(sample(1:n(), 0.8*n())) %>%
  ungroup()%>% select(-Treatment)%>% as.matrix() %>% as.vector()

val.ID = as.character(y_metadata[!y_metadata$ID %in% cal.ID, "ID"])

cat("n_cal =", length(cal.ID), " n_val =", length(val.ID), "\n")

## ---- feature selection (Rmd chunk, verbatim) ----
pre_cal = input_for_RReliefF(X_IDdata = X_IDdata, y_metadata = y_metadata,
                            cal.ID = cal.ID, y_name = 'N',
                            num_X_names = num_X_names)

RReliefF = RReliefF_algorithm(pre_cal = pre_cal,
                                y_name = 'N', num_X_names = num_X_names)
RReliefF_star =  RReliefF[order(RReliefF$value, decreasing = TRUE)[1:400], ]
print(tail(RReliefF_star))

## ---- cal/val data sets + hist plot (Rmd chunk) ----
png(file.path(out_dir, "01_cal_val_hist.png"), width = 1800, height = 900, res = 200)
cal_val_list = cal_val_data_sets(cal.ID = cal.ID, val.ID = val.ID,
                                   X_IDdata = X_IDdata, y_metadata = y_metadata,
                                   X_attr = RReliefF_star,
                                   y_name = 'N', num_X_names = num_X_names,
                                   plot = TRUE, y_unit = " (%)")
dev.off()

cal.plsr.data = cal_val_list[[1]]
val.plsr.data = cal_val_list[[2]]

## ---- spectra in cal/val (Rmd chunk) ----
cal.plsr.long = sel_wv_plotdata(cal.plsr.data)
val.plsr.long = sel_wv_plotdata(val.plsr.data)

sel_wv_ref_min = range(cal.plsr.long$value, val.plsr.long$value)[1]
sel_wv_ref_max = range(cal.plsr.long$value, val.plsr.long$value)[2]

summary_cal_long = quantile_sel_ref(data = cal.plsr.long, 0.025, 0.975)
summary_val_long = quantile_sel_ref(data = val.plsr.long, 0.025, 0.975)

cal_ref_95perc = ggplot()+
  geom_point(aes(x=variable, y=value, group=ID), data=cal.plsr.long,
             size=0.3, color="grey")+
  geom_point(aes(x=wv, y=value, col=variable), data = summary_cal_long,
             size=0.5, show.legend = FALSE)+
  scale_color_manual(values = summary_ref_col)+
  scale_y_continuous(breaks = seq(sel_wv_ref_min, sel_wv_ref_max, length.out = 3),
                     labels = round(seq(sel_wv_ref_min, sel_wv_ref_max, length.out = 3), 3),
                     limits = c(sel_wv_ref_min, sel_wv_ref_max)) +
  labs(title = "N calibration dataset", x= "Wavelength (nm)", y = "Reflectance (%)") +
  theme_bw(base_size=18) +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
        )

val_ref_95perc = ggplot()+
  geom_point(aes(x=variable, y=value, group=ID), data=val.plsr.long,
             size=0.3, color="grey")+
  geom_point(aes(x=wv, y=value, col=variable), data = summary_val_long,
             size=0.5, show.legend = FALSE)+
  scale_color_manual(values = summary_ref_col)+
  scale_y_continuous(breaks = seq(sel_wv_ref_min, sel_wv_ref_max, length.out = 3),
                     labels = round(seq(sel_wv_ref_min, sel_wv_ref_max, length.out = 3), 3),
                     limits = c(sel_wv_ref_min, sel_wv_ref_max)) +
  labs(title = "N validation dataset", x= "Wavelength (nm)", y = "Reflectance (%)") +
  theme_bw(base_size=18) +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
        )

ref_95perc = ggarrange(cal_ref_95perc, val_ref_95perc, nrow = 2)
ggsave(filename = file.path(out_dir, "02_spectra_cal_val.png"),
       plot = ref_95perc, width = 9, height = 5, units = "in", dpi = 200)

## ---- selected wv (Rmd chunk) ----
full_spectra_calID = melt(X_IDdata[X_IDdata$ID %in% cal.ID, -1]) #remove ID

full_spectra_calID_summary = full_spectra_calID %>%
  group_by(variable) %>%
  summarize(mean = mean(value),
            lo = quantile(value, 0.025),
            hi = quantile(value, 0.975)) %>%
  mutate(wv = as.numeric(as.character(variable)))

selected_wv = ggplot()+
  geom_vline(aes(xintercept=variable), data=cal.plsr.long, color = "green") +
  geom_ribbon(aes(x = wv, ymin = lo, ymax=hi),
              fill = 'grey70', data = full_spectra_calID_summary) +
  geom_line(aes(x = wv, y = mean), data = full_spectra_calID_summary) +
  theme_bw()+
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
        ) +
  labs(x='Wavelength (nm)', y = 'Reflectance (%)')
ggsave(filename = file.path(out_dir, "03_selected_wv.png"),
       plot = selected_wv, width = 7, height = 5, units = "cm", dpi = 300)

## ---- number of components CV (Rmd chunk) ----
plsr.ini <- plsr(N~Spectra, scale=FALSE,
                 validation="LOO",trace=FALSE,data=cal.plsr.data)
maxComps <- 15

png(file.path(out_dir, "04_ncomps_rmsep.png"), width = 4, height = 5,
    units = "in", res = 200)
nComps = min_selectNcomp(plsr.ini, plot = TRUE,
                lwd=1.6, xlim = c(0, maxComps), pch=21, col="black",
                bg = "grey70", cex.axis = 2, cex.lab = 2, font.lab = 2, font = 2)
box(lwd=2.2)
dev.off()

cat("selected nComps =", nComps, "\n")

## ---- plsr output (Rmd chunk) ----
plsr.out <- plsr(N~Spectra, scale=FALSE, ncomp = nComps,
                 validation="LOO",trace=FALSE,data=cal.plsr.data, jackknife=TRUE)

plsr.output.data = plsr_results(model=plsr.out,
                                      nComps=nComps,
                                      cal.plsr.data=cal.plsr.data,
                                      val.plsr.data=val.plsr.data,
                                      y_name='N')
print(plsr.output.data$mm)

cal.plsr.output = plsr.output.data$cal.plsr.output
val.plsr.output = plsr.output.data$val.plsr.output

## ---- jackknife coef (Rmd chunk `jackknife coef`) ----
## f.coef.valid embedded verbatim from plantphys/spectratrait @ d159c5d (GPL-3.0)
f.coef.valid <- function(plsr.out, data_plsr, ncomp, inVar) {
  ## Only work in the case where center=TRUE in the plsr model
  B <- plsr.out$validation$coefficients[, , ncomp,, drop = FALSE]
  dB <- dim(B)
  dB[1] <- dB[1] + 1
  dnB <- dimnames(B)
  dnB[[1]] <- c("(Intercept)", dnB[[1]])
  BInt <- array(dim = dB, dimnames = dnB)
  BInt[-1, , ,] <- B
  nseg=dB[[4]]
  for (i in 1:nseg){
    Y<-data_plsr[,inVar]
    Y<-Y[-plsr.out$validation$segments[[i]]]
    Ymeans<-mean(Y)
    X<-data_plsr$Spectra
    X<-X[-plsr.out$validation$segments[[i]],]
    Xmeans<-colMeans(X)
    BInt[1, , ,i] <- Ymeans - Xmeans %*% B[, , , i]
  }
  B <- BInt
  return(B)
}

jackknife = f.coef.valid(plsr.out,
                                             cal.plsr.data,
                                             nComps, 'N')
intercepts = jackknife[1,,,]
coefs = jackknife[-1,,,]
pred_jackknife = matrix(rep(intercepts, length(val.plsr.data[,'N'])), byrow = TRUE,
                          ncol = length(intercepts))+
                   val.plsr.data$Spectra %*% coefs

val.plsr.output$uci = apply(pred_jackknife, 1, quantile, 0.975)
val.plsr.output$lci = apply(pred_jackknife, 1, quantile, 0.025)

model_sd_mean = apply(pred_jackknife, 1, sd)
model_sd_res = sd(val.plsr.output$Residuals)
model_totsd = sqrt(model_sd_mean^2 + model_sd_res^2)

val.plsr.output$upi = val.plsr.output$uci + 1.96*model_totsd
val.plsr.output$lpi = val.plsr.output$lci - 1.96*model_totsd
print(head(val.plsr.output))

## ---- scatter/residual plots (Rmd chunks) ----
rng = range(val.plsr.output$upi, val.plsr.output$lpi,
            val.plsr.output$Observed, val.plsr.output$Predicted,
            cal.plsr.output$Observed, cal.plsr.output$Predicted)
text_xloc_n <- seq(round(rng[1]), round(rng[2]), length = 10)[4]
text_yloc_n <- (rng[2]-rng[1])/60*58+rng[1]
text_xloc_mm <- seq(round(rng[1]), round(rng[2]), length = 10)[7]
text_y_R2 <- (rng[2]-rng[1])/10*2.1+rng[1]
text_y_RMSEP <- (rng[2]-rng[1])/10*1.2+rng[1]
text_y_NRMSEP <- (rng[2]-rng[1])/10*0.3+rng[1]

cal_scatter_plot <- ggplot(cal.plsr.output, aes(x=Observed, y=Predicted)) +
  geom_abline(intercept = 0, slope = 1, size=0.3) +
  geom_point(aes(col=Subpop, shape = Treatment)) +
  scale_color_manual(values = subpop_col)+
  scale_shape_manual(values = treatment_shape)+
  scale_x_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  scale_y_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  theme_bw(base_size = 6) +
  labs(x="\n Observed: Nitrogen (%)",
       y="Predicted: Nitrogen (%)") +
  theme(legend.position="none") +
 annotate("text", size = 2 , x= text_xloc_mm, y = c(text_y_R2, text_y_RMSEP), parse = TRUE,
           label = c(paste("R^{2} == ", plsr.output.data$mm[1,"R2"]),
                     paste("RMSEP ==", plsr.output.data$mm[1,"RMSEP"]))) +
 annotate("text", size = 2 , x= text_xloc_mm, y = text_y_NRMSEP,
           label = paste0("%RMSEP = ", plsr.output.data$mm[1,"NRMSEP"])) +
 annotate("text", x = text_xloc_n, y = text_yloc_n, parse = TRUE,
           label = paste0("Cal. (italic(n) == ", length(cal.ID), ")"), size = 2)

cal_resid_plot <- ggplot(cal.plsr.output, aes(x=Observed, y=Residuals)) +
  geom_hline(yintercept = 0, size = 0.3)+
  geom_point(aes(col=Subpop, shape = Treatment)) +
  scale_color_manual(values = subpop_col)+
  scale_shape_manual(values = treatment_shape)+
  theme_bw(base_size = 6)+
  theme(legend.position="none") +
  labs(x="\n Observed: Nitrogen (%)",
       y="Residuals")

val_CI_plot <- ggplot(val.plsr.output, aes(x=Observed, y=Predicted)) +
  geom_abline(intercept = 0, slope = 1, size=0.3) +
  geom_point(aes(col=Subpop, shape = Treatment)) +
  scale_shape_manual(values = treatment_shape)+
  geom_errorbar(aes(ymin = lci, ymax = uci,col=Subpop), width=0.2, size = 0.1) +
  scale_color_manual(values = subpop_col)+
  scale_x_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  scale_y_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  theme_bw(base_size = 6) +
  labs(x="\n Observed: Nitrogen (%)",
       y="Predicted: Nitrogen (%)") +
  theme(legend.position="none") +
  annotate("text", size = 2 , x= text_xloc_mm, y = c(text_y_R2, text_y_RMSEP), parse = TRUE,
           label = c(paste("R^{2} == ", plsr.output.data$mm[2,"R2"]),
                     paste("RMSEP ==", plsr.output.data$mm[2,"RMSEP"]))) +
  annotate("text", size = 2 , x= text_xloc_mm, y = text_y_NRMSEP,
           label = paste0("%RMSEP = ", plsr.output.data$mm[2,"NRMSEP"])) +
  annotate("text", x = text_xloc_n, y = text_yloc_n, parse = TRUE,
           label = paste0("Val. (italic(n) == ", length(val.ID), ")"), size = 2)

val_PI_plot <- ggplot(val.plsr.output, aes(x=Observed, y=Predicted)) +
  geom_abline(intercept = 0, slope = 1, size=0.3) +
  geom_point(aes(col=Subpop, shape = Treatment)) +
  scale_shape_manual(values = treatment_shape)+
  geom_errorbar(aes(ymin = lpi, ymax = upi, col=Subpop), width=0.2, size = 0.1) +
  scale_color_manual(values = subpop_col)+
  scale_x_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  scale_y_continuous(breaks = seq(round(rng[1]), round(rng[2]), length = 5),
                     labels = seq(round(rng[1]), round(rng[2]), length = 5),
                     limits = c(rng[1], rng[2]))+
  theme_bw(base_size = 6) +
  labs(x="\n Observed: Nitrogen (%)",
       y="Predicted: Nitrogen (%)") +
  theme(legend.position="none") +
  annotate("text", size = 2 , x= text_xloc_mm, y = c(text_y_R2, text_y_RMSEP), parse = TRUE,
           label = c(paste("R^{2} == ", plsr.output.data$mm[2,"R2"]),
                     paste("RMSEP ==", plsr.output.data$mm[2,"RMSEP"]))) +
  annotate("text", size = 2 , x= text_xloc_mm, y = text_y_NRMSEP,
           label = paste0("%RMSEP = ", plsr.output.data$mm[2,"NRMSEP"])) +
  annotate("text", x = text_xloc_n, y = text_yloc_n, parse = TRUE,
           label = paste0("Val. (italic(n) == ", length(val.ID), ")"), size = 2)

val_resid_plot <- ggplot(val.plsr.output, aes(x=Observed, y=Residuals)) +
  geom_hline(yintercept = 0, size = 0.3)+
  geom_point(aes(col=Subpop, shape = Treatment)) +
  scale_color_manual(values = subpop_col)+
  scale_shape_manual(values = treatment_shape)+
  theme_bw(base_size = 6)+
  theme(legend.position="none") +
  labs(x="\n Observed: Nitrogen (%)",
       y="Residuals")

ggsave(filename = file.path(out_dir, "05_cal_scatter.png"), plot = cal_scatter_plot,
       width = 5, height = 5, units = "cm", dpi = 300)
ggsave(filename = file.path(out_dir, "06_cal_resid.png"), plot = cal_resid_plot,
       width = 3.5, height = 3.5, units = "cm", dpi = 300)
ggsave(filename = file.path(out_dir, "07_val_CI.png"), plot = val_CI_plot,
       width = 5, height = 5, units = "cm", dpi = 300)
ggsave(filename = file.path(out_dir, "08_val_PI.png"), plot = val_PI_plot,
       width = 3.5, height = 3.5, units = "cm", dpi = 300)
ggsave(filename = file.path(out_dir, "09_val_resid.png"), plot = val_resid_plot,
       width = 3.5, height = 3.5, units = "cm", dpi = 300)

## ---- outputs (Rmd chunk `save plots and data`) ----
write.csv(x = RReliefF, file = file.path(out_dir, "N_RReliefF_all.csv"), row.names = FALSE)
write.csv(x = cal.plsr.output, file = file.path(out_dir, "N.cal.plsr.output.csv"), row.names = FALSE)
write.csv(x = val.plsr.output, file = file.path(out_dir, "N.val.plsr.output.csv"), row.names = FALSE)
write.csv(x = plsr.output.data$mm, file = file.path(out_dir, "N_metrics.csv"), row.names = FALSE)

cat("DONE nComps=", nComps, " cal_R2=", plsr.output.data$mm[1,"R2"],
    " val_R2=", plsr.output.data$mm[2,"R2"],
    " val_RMSEP=", plsr.output.data$mm[2,"RMSEP"], "\n")


## 5. Run the pipeline
Launches `Rscript /content/n400_plsr.R` (CPU). Streams stdout/stderr; prints the split sizes, the
RReliefF tail, the selected number of components, the calibration/validation metric table, and a
final DONE line. 日本語: R パイプラインを実行し、指標（R², RMSEP, %RMSEP）を表示します。

In [ ]:
import time
t0 = time.time()
proc = subprocess.run(["Rscript", "/content/n400_plsr.R"],
                      capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"R pipeline failed with exit code {proc.returncode}")
print(f"R pipeline finished in {time.time()-t0:.0f}s")

## 6. Results
Shows the metric table (`N_metrics.csv`, R²/RMSEP/NRMSEP for cal and val), the predicted-vs-observed
scatter plots with jackknife intervals, and the selected-wavelength figure. Compare the validation
row with the paper's N-model reference (R² = 0.797, RMSEP = 0.264): exact values may differ modestly
because the calibration/validation split resamples `set.seed(1)`-governed sampling on the actual
column layout, and the authors noted their annotated metrics vary with run timing.
日本語: 検証指標を論文の参照値（val R² 0.797, RMSEP 0.264）と比較します。乱数分割により多少の差は許容されます。

In [ ]:
import pandas as pd
from IPython.display import Image, display

metrics = pd.read_csv("/content/outputs/N_metrics.csv")
print(metrics.to_string(index=False))
display(Image(filename="/content/outputs/02_spectra_cal_val.png"))
display(Image(filename="/content/outputs/04_ncomps_rmsep.png"))
display(Image(filename="/content/outputs/05_cal_scatter.png"))
display(Image(filename="/content/outputs/07_val_CI.png"))
display(Image(filename="/content/outputs/03_selected_wv.png"))


## 7. Interpretation & validation record
- **What ran:** the author's own R pipeline (commit `b6b6220`) for the N (%), 400-wavelength, side-view
  Week-9 model, on the paper's PURR 4079 v1 data, executed on **CPU** in Colab.
- **What this verifies:** the pipeline executes end-to-end and produces calibration/validation metrics,
  selected wavelengths, and jackknife intervals. It does **not** verify dataset-level accuracy claims
  or the paper's other 25 trait pipelines, SVM classification, or transfer/hyperparameter analyses.
- **Adaptations:** `/content` paths; PNG outputs; %RMSEP annotations computed from the model; embedded
  `f.coef.valid` (GPL-3.0) instead of a GitHub package install. No algorithmic changes.
- **References:** paper doi:10.3389/fpls.2023.1229161 · code
  [To-Chia/rice_imaging_ms](https://github.com/To-Chia/rice_imaging_ms) @ b6b6220 · data
  [PURR 4079 v1](https://purr.purdue.edu/publications/4079/1) doi:10.4231/SY0A-W347 (CC BY 4.0) ·
  [spectratrait](https://github.com/plantphys/spectratrait) @ d159c5d (GPL-3.0).

日本語: 本ノートブックは著者の R パイプライン（leaf N 予測、400 波長）を CPU 上でそのまま実行した
部分的な再現デモです。論文のデータセット全体の主張や他の形質パイプラインの検証ではありません。